<h2 style="color:#2E86C1; margin-bottom:6px;">
  01 – Traditional Baseline: TF-IDF + One-vs-Rest Logistic Regression
</h2>

<p style="font-size:16px; margin-top:4px;">
  <strong>Overview:</strong>
  This notebook builds a first baseline model for multi-label emotion classification
  using a traditional machine learning pipeline.
</p>

<p style="font-size:16px;">
  <strong>Goal:</strong>
  Predict five possible emotions for each input text using TF-IDF features and a
  One-vs-Rest Logistic Regression classifier:
</p>

<ul style="font-size:16px; margin-left:18px;">
  <li>Anger</li>
  <li>Fear</li>
  <li>Joy</li>
  <li>Sadness</li>
  <li>Surprise</li>
</ul>

<p style="font-size:16px;">
  This model is simple, interpretable, and establishes an initial
  <strong>Macro F1</strong> score before moving on to neural and pretrained
  transformer-based models.
</p>


In [2]:
#wandb login check to make sure everything goes smooth

from kaggle_secrets import UserSecretsClient
import wandb

# Load API key from Kaggle Secrets
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("WANDB_API_KEY")

# Login to wandb
wandb.login(key=api_key)

print("W&B login successful!")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sharmilam-official (sharmila-m) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login successful!


In [3]:
# 01 – TF-IDF + One-vs-Rest Logistic Regression (Baseline)

import pandas as pd
import wandb
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, accuracy_score

# -------------------------------------
# 0. Initialize W&B
# -------------------------------------
wandb.init(
    project="24ds2000048-t32025",
    entity="sharmila-m",
    name="01_tfidf_logistic_regression_ovr",
    config={
        "max_features": 5000,
        "ngram_range": (1, 2),
        "stop_words": "english",
        "model": "LogisticRegression",
        "max_iter": 200,
        "test_size": 0.2,
        "threshold": 0.25
    }
)

config = wandb.config

# -------------------------------------
# 1. Load data
# -------------------------------------
train = pd.read_csv("/kaggle/input/2025-sep-dl-gen-ai-project/train.csv")
test = pd.read_csv("/kaggle/input/2025-sep-dl-gen-ai-project/test.csv")

# -------------------------------------
# 2. Split features and labels
# -------------------------------------
X = train["text"]
y = train[["anger", "fear", "joy", "sadness", "surprise"]]

# -------------------------------------
# 3. Train/Validation Split
# -------------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=config.test_size,
    random_state=42
)

# -------------------------------------
# 4. TF-IDF Vectorizer
# -------------------------------------
tfidf = TfidfVectorizer(
    max_features=config.max_features,
    ngram_range=tuple(wandb.config.ngram_range),
    stop_words=config.stop_words
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(test["text"])

# -------------------------------------
# 5. Train Logistic Regression (OVR)
# -------------------------------------
model = OneVsRestClassifier(
    LogisticRegression(max_iter=config.max_iter)
)

model.fit(X_train_tfidf, y_train)

# -------------------------------------
# 6. Validate
# -------------------------------------
val_preds = model.predict(X_val_tfidf)
val_f1 = f1_score(y_val, val_preds, average="macro")
val_acc = accuracy_score(y_val, val_preds)

print("Validation Macro F1 Score:", val_f1)
print("Validation Accuracy:", val_acc)

# Log metrics to W&B
wandb.log({
    "val_f1_macro": val_f1,
    "val_accuracy": val_acc
})

# -------------------------------------
# 7. Train on full data (keep same vocabulary)
# -------------------------------------
full_tfidf = tfidf.transform(X)
model.fit(full_tfidf, y)

wandb.finish()

Validation Macro F1 Score: 0.43992646911373995
Validation Accuracy: 0.32430453879941434


val_accuracy,▁
val_f1_macro,▁
val_accuracy,0.3243
val_f1_macro,0.43993


In [4]:
# -------------------------------------
# 8. Predict on test
# -------------------------------------
probs = model.predict_proba(X_test_tfidf)
test_preds = (probs > config.threshold).astype(int)


# -------------------------------------
# 9. Create submission file
# -------------------------------------
submission = pd.concat(
    [
        test["id"],
        pd.DataFrame(
            test_preds,
            columns=["anger", "fear", "joy", "sadness", "surprise"]
        )
    ],
    axis=1
)

submission.to_csv("submission.csv", index=False)
print("Generated Submission.csv file successfully")

Generated Submission.csv file successfully


<hr/>

<h3 style="color:#1F618D;">
  Results & Runtime Summary
</h3>

<p style="font-size:16px;">
  <strong>Validation Metric (Internal):</strong><br/>
  Macro F1 Score: <strong>0.4399</strong>
</p>

<p style="font-size:16px;">
  <strong>Kaggle Public Leaderboard Score:</strong><br/>
  Macro F1 Score: <strong>0.612</strong>
</p>

<p style="font-size:16px;">
  <strong>Runtime Environment:</strong>
</p>

<ul style="font-size:16px; line-height:1.6;">
  <li><strong>Platform:</strong> Kaggle Notebook</li>
  <li><strong>Hardware:</strong> CPU (no GPU required)</li>
  <li><strong>Runtime:</strong> 38s</li>
  <li><strong>Frameworks:</strong> scikit-learn, pandas</li>
</ul>

<p style="font-size:15px; color:#555;">
  <em>This traditional baseline model provides a strong and interpretable
  reference point for comparing more complex neural and transformer-based models.</em>
</p>
